# Ultrasonic UV Panel Monitoring — Training Notebook

Train a **MobileNetV2** classifier on **linear spectrograms** from the
**Dodotronic Ultramic 384K EVO** (384 kHz sample rate, 0–192 kHz frequency range).

### Key changes from the original audible-range script
| Setting | Old (audible) | New (ultrasonic) | Why |
|---------|--------------|------------------|-----|
| Sample rate | 48,000 Hz | **384,000 Hz** | Match the Ultramic 384K EVO |
| Frequency range | 50–20,000 Hz | **0–192,000 Hz** | Full Nyquist range |
| Spectrogram | Mel | **Linear** | Mel compresses ultrasonics; linear preserves them |
| Model | ResNet18 (11.7 M) | **MobileNetV2 (3.4 M)** | Lightweight for Raspberry Pi 4 inference |
| Export | fastai .pkl only | **TorchScript .pt + meta.json** | No fastai needed on the Pi |


In [ ]:
# Imports
from pathlib import Path
import random
import json

import torch
import torch.nn.functional as F
import torchaudio
import soundfile as sf
import numpy as np

from fastai.vision.all import *

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


## Configuration

Update `PROJECT_ROOT` and `DATA_ROOT` to match your local paths.
Everything else is tuned for the Ultramic 384K EVO.


In [ ]:
# Project paths — EDIT THESE to match your setup
PROJECT_ROOT = Path('.')
DATA_ROOT    = PROJECT_ROOT / 'data' / 'ultrasonic'

# Class folders (must match folder names under DATA_ROOT)
CLASSES = ['normal', 'abnormal', 'abnormal2', 'pulsing_unstable']


In [ ]:
# Ultramic 384K EVO audio settings
SAMPLE_RATE = 384_000          # 384 kHz native sample rate
SEGMENT_SEC = 1.0              # 1-second input windows
NUM_SAMPLES = int(SAMPLE_RATE * SEGMENT_SEC)  # 384,000 samples per segment

# Linear spectrogram settings (NOT mel — linear preserves ultrasonic detail)
N_FFT       = 4096             # FFT size -> 2049 freq bins covering 0-192 kHz
WIN_LENGTH  = 2048             # analysis window
HOP_LENGTH  = 1024             # hop -> ~375 time frames per second

# Full frequency range: 0 Hz to Nyquist (192,000 Hz)
F_MIN = 0.0
F_MAX = SAMPLE_RATE / 2        # 192,000 Hz

# Model input image size (MobileNetV2 default)
IMG_SIZE = 224

# Training hyperparameters
BATCH_SIZE = 32
VALID_PCT  = 0.2
SEED       = 42

print(f'Sample rate  : {SAMPLE_RATE:,} Hz')
print(f'Freq range   : {F_MIN:,.0f} - {F_MAX:,.0f} Hz')
print(f'Segment      : {SEGMENT_SEC}s = {NUM_SAMPLES:,} samples')
print(f'Spectrogram  : {N_FFT//2+1} freq bins x ~{NUM_SAMPLES//HOP_LENGTH} time frames')
print(f'Model input  : {IMG_SIZE}x{IMG_SIZE} pixels')


## Data Sanity Check

In [ ]:
# Verify data paths exist
print('DATA_ROOT:', DATA_ROOT.resolve())
print('Exists?  ', DATA_ROOT.exists())
print()

for cls in CLASSES:
    cls_dir = DATA_ROOT / cls
    n_files = len(list(cls_dir.glob('*.wav'))) if cls_dir.exists() else 0
    status = 'OK' if cls_dir.exists() else 'MISSING'
    print(f'  {cls:20s}  {status:7s}  {n_files} wav files')


In [ ]:
# Load and inspect one WAV file
test_wav = None
for cls in CLASSES:
    cls_dir = DATA_ROOT / cls
    if cls_dir.exists():
        wavs = list(cls_dir.glob('*.wav'))
        if wavs:
            test_wav = wavs[0]
            break

if test_wav:
    data, sr = sf.read(str(test_wav))
    print(f'File       : {test_wav.name}')
    print(f'Sample rate: {sr:,} Hz')
    print(f'Duration   : {len(data)/sr:.3f}s')
    print(f'Shape      : {data.shape}')
else:
    print('No WAV files found - add data to DATA_ROOT before training.')


## Linear Spectrogram Transform

Uses `torchaudio.transforms.Spectrogram` (**linear**, NOT mel) so ultrasonic
frequencies above 20 kHz are preserved with full resolution. The spectrogram is:
1. Converted to dB scale
2. Quantile-normalised (10th–99th percentile) per sample
3. Resized to 224×224 for the MobileNetV2 input
4. Replicated to 3 channels (pretrained CNN expects RGB)


In [ ]:
# AudioToSpec transform — linear spectrogram for ultrasonic frequencies

class AudioToSpec(Transform):
    """Load WAV -> crop/pad -> linear spectrogram -> dB -> normalise -> 3-ch image."""

    def __init__(self, sample_rate=SAMPLE_RATE, n_fft=N_FFT,
                 win_length=WIN_LENGTH, hop_length=HOP_LENGTH,
                 img_size=IMG_SIZE, seconds=SEGMENT_SEC):
        super().__init__()
        self.sample_rate = sample_rate
        self.num_samples = int(sample_rate * seconds)
        self.img_size = img_size

        # Linear spectrogram (NOT mel)
        self.spectrogram = torchaudio.transforms.Spectrogram(
            n_fft=n_fft,
            win_length=win_length,
            hop_length=hop_length,
            power=2.0,
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype='power')
        self.resamplers = {}

    def _resample(self, waveform, sr):
        if sr == self.sample_rate:
            return waveform
        if sr not in self.resamplers:
            self.resamplers[sr] = torchaudio.transforms.Resample(
                orig_freq=sr, new_freq=self.sample_rate
            )
        return self.resamplers[sr](waveform)

    def encodes(self, fn: Path):
        # Load WAV
        data, sr = sf.read(str(fn))
        waveform = torch.from_numpy(data).float()

        # Ensure mono [1, T]
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)
        else:
            waveform = waveform.T
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample if needed
        waveform = self._resample(waveform, sr)

        # Crop / pad to fixed length
        if waveform.shape[1] > self.num_samples:
            waveform = waveform[:, :self.num_samples]
        else:
            pad = self.num_samples - waveform.shape[1]
            waveform = F.pad(waveform, (0, pad))

        # Linear spectrogram -> dB
        spec = self.spectrogram(waveform)
        spec_db = self.to_db(spec)

        # Quantile normalization (adaptive per-sample)
        lo = torch.quantile(spec_db, 0.10)
        hi = torch.quantile(spec_db, 0.99)
        spec_db = spec_db.clamp(lo, hi)
        spec_img = (spec_db - lo) / (hi - lo + 1e-9)

        # Resize to model input size
        spec_img = spec_img.unsqueeze(0)   # [1, 1, H, W]
        spec_img = F.interpolate(
            spec_img, size=(self.img_size, self.img_size),
            mode='bilinear', align_corners=False
        )
        spec_img = spec_img.squeeze(0)     # [1, H, W]

        # 3-channel for pretrained CNN
        img = spec_img.repeat(3, 1, 1)
        return TensorImage(img)


## DataBlock and DataLoaders

In [ ]:
# Build DataLoaders

def get_audio_files(path):
    return get_files(path, extensions=['.wav'], recurse=True)

audio_tfms = AudioToSpec(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    win_length=WIN_LENGTH,
    hop_length=HOP_LENGTH,
    img_size=IMG_SIZE,
    seconds=SEGMENT_SEC,
)

dblock = DataBlock(
    blocks=(TransformBlock(type_tfms=audio_tfms), CategoryBlock),
    get_items=get_audio_files,
    get_y=parent_label,
    splitter=RandomSplitter(valid_pct=VALID_PCT, seed=SEED),
)

dls = dblock.dataloaders(DATA_ROOT, bs=BATCH_SIZE, drop_last=False)
dls


In [ ]:
# Sanity check: visualise spectrogram batch
dls.show_batch(max_n=9, figsize=(10, 10))


In [ ]:
# Dataset statistics
print(f'Train samples: {len(dls.train_ds)}')
print(f'Valid samples: {len(dls.valid_ds)}')
print(f'Train batches: {len(dls.train)}')
print(f'Valid batches: {len(dls.valid)}')
print(f'Classes: {dls.vocab}')


## Model Training — MobileNetV2

MobileNetV2 is ~3x lighter than ResNet18 and purpose-built for edge devices
like the Raspberry Pi 4. We use ImageNet-pretrained weights and fine-tune
on our ultrasonic spectrograms.

> **Alternative:** For even lighter models, try `'mobilenetv3_small_100'`
> via timm (2.5M params), or keep `resnet18` if accuracy is more important
> than inference speed.


In [ ]:
# Create learner with MobileNetV2
from torchvision.models import mobilenet_v2

learn = vision_learner(dls, mobilenet_v2, metrics=accuracy)
learn


In [ ]:
# Learning rate finder
lr_min, lr_steep = learn.lr_find()
print(f'LR suggestions - min: {lr_min:.2e}, steep: {lr_steep:.2e}')


In [ ]:
# Train (adjust epochs and lr as needed)
EPOCHS = 10
LR = 3e-4

learn.fit_one_cycle(EPOCHS, LR)


In [ ]:
# Plot training loss
learn.recorder.plot_loss()


## Evaluation

In [ ]:
# Confusion matrix
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(6, 6))


In [ ]:
# Top losses — inspect the worst predictions
interp.plot_top_losses(6, nrows=2, figsize=(10, 5))


In [ ]:
# Manual predictions on sample files from each class
for cls in CLASSES:
    cls_dir = DATA_ROOT / cls
    if cls_dir.exists():
        wav = next(cls_dir.glob('*.wav'), None)
        if wav:
            pred_class, pred_idx, probs = learn.predict(wav)
            print(f'{cls:20s} -> predicted: {pred_class}  (probs: {probs.numpy().round(3)})')


## Export Model

Exports three artefacts:
1. **`ultrasonic_model.pt`** — TorchScript model (deploy to Pi 4, no fastai needed)
2. **`meta.json`** — spectrogram config (inference script reads this)
3. **`ultrasonic_model_fastai.pkl`** — fastai learner backup


In [ ]:
# Export TorchScript model + meta.json

# --- TorchScript ---
model_cpu = learn.model.eval().cpu()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
traced = torch.jit.trace(model_cpu, dummy_input)

ts_path = PROJECT_ROOT / 'ultrasonic_model.pt'
traced.save(str(ts_path))
print(f'TorchScript model saved: {ts_path}')

# --- meta.json (inference config — matches preprocessing exactly) ---
meta = {
    'classes': list(dls.vocab),
    'sample_rate': SAMPLE_RATE,
    'segment_sec': SEGMENT_SEC,
    'n_fft': N_FFT,
    'win_length': WIN_LENGTH,
    'hop_length': HOP_LENGTH,
    'img_size': IMG_SIZE,
}

meta_path = PROJECT_ROOT / 'meta.json'
meta_path.write_text(json.dumps(meta, indent=2))
print(f'Metadata saved: {meta_path}')
print(json.dumps(meta, indent=2))

# --- fastai export (backup) ---
learn.export(PROJECT_ROOT / 'ultrasonic_model_fastai.pkl')
print(f'fastai export saved: {PROJECT_ROOT / "ultrasonic_model_fastai.pkl"}')


In [ ]:
# Verify TorchScript model loads and produces correct output shape
loaded = torch.jit.load(str(ts_path), map_location='cpu')
loaded.eval()

test_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    out = loaded(test_input)

print(f'Output shape: {out.shape}')
print(f'Num classes : {out.shape[1]}')
print(f'Classes     : {list(dls.vocab)}')
print()
print('Model is ready to deploy to Raspberry Pi 4!')
print('Copy these files to the Pi:')
print(f'  1. {ts_path}')
print(f'  2. {meta_path}')
print(f'  3. ultrasonic_infer.py')
